In [ ]:
!pip install -r requirements.txt

# Classication

In [ ]:
from base_conformal import ConformalClassifierMIA
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris, load_wine, load_breast_cancer
from sklearn.model_selection import train_test_split
from crepes import ConformalClassifier
from crepes.extras import hinge
import numpy as np
from pprint import pprint

RANDOM_SEED = 0
epsilon = 0.1

X, y = load_iris(return_X_y=True)
X = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
model = RandomForestClassifier(n_estimators=10,random_state=RANDOM_SEED)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)
X_train, X_calib, y_train, y_calib = train_test_split(X_train, y_train, test_size=0.2, random_state=RANDOM_SEED)

model.fit(X_train, y_train)
probs_calib = model.predict_proba(X_calib)
probs_test = model.predict_proba(X_test)

cp = ConformalClassifier()
alphas_cal = hinge(probs_calib,model.classes_, y_calib)
cp.fit(alphas_cal)
alphas_test = hinge(probs_test)


cp_mia = ConformalClassifierMIA()
cp_mia.fit_mia(X_train, y_train)
alphas_calib_mica = cp_mia.alpha_weighting(X_calib, probs_calib,classes=model.classes_, y=y_calib,hinge_margin="complex_hinge")
cp_mia.fit(alphas_calib_mica)
alphas_test_mica = cp_mia.alpha_weighting(X_test, probs_test, hinge_margin="complex_hinge")

#print("CP:", cp.predict_set(alphas=alphas_test))
#print("CP-MIA:", cp_mia.predict_set(alphas=alphas_test_mica))

pprint(f"CP: {cp.evaluate(alphas_test, model.classes_, y_test,confidence=1-epsilon)}")
pprint(f"CP-MIA: {cp_mia.evaluate(alphas_test_mica, model.classes_, y_test,confidence=1-epsilon)}")

def UCEM(predict_sets,y_true,way = "accuracy_accuracy"):
    n_classes = predict_sets.shape[1]
    predict_sets = np.array(predict_sets)
    set_sizes = np.sum(predict_sets, axis=1)

    conf_matrix = np.zeros((n_classes+1, 2, 2), dtype=int)
    for c in range(n_classes):
        for i in range(len(y_true)):
            pred = predict_sets[i, c]
            actual = int(y_true[i] == c)
            if actual == 1 and pred == 1:
                conf_matrix[c, 1, 1] += 1  # TP
            elif actual == 0 and pred == 1:
                conf_matrix[c, 0, 1] += 1  # FP
            elif actual == 1 and pred == 0:
                conf_matrix[c, 1, 0] += 1  # FN
            else:
                conf_matrix[c, 0, 0] += 1  # TN
    #consider the empty set as an additional "class" that will be a false negative if the predicted set is empty and ... if it predicts not empty
    conf_matrix[n_classes, 1, 0] = np.sum((set_sizes == 0) & (y_true >= 0))  # FN for empty set
    conf_matrix[n_classes, 0, 1] = np.sum((set_sizes > 0) & (y_true < 0))  # FP for empty set
    conf_matrix[n_classes, 0, 0] = np.sum((set_sizes == 0) & (y_true < 0))  # TN for empty set
    conf_matrix[n_classes, 1, 1] = np.sum((set_sizes > 0) & (y_true >= 0))  # TP for empty set

    metric = None
    if way == "f1_accuracy":
        metric_vector = np.zeros(n_classes+1)
        for c in range(n_classes):
            TP = conf_matrix[c, 1, 1]
            FP = conf_matrix[c, 0, 1]
            FN = conf_matrix[c, 1, 0]
            metric_vector[c] = 2 * TP / (2 * TP + FP + FN) if (2 * TP + FP + FN) > 0 else 0
        metric_vector[n_classes] = conf_matrix[n_classes, 1, 1] / (conf_matrix[n_classes, 1, 1] + conf_matrix[n_classes, 0, 1]) if (conf_matrix[n_classes, 1, 1] + conf_matrix[n_classes, 0, 1]) > 0 else 0
    elif way == "accuracy_accuracy":
        metric_vector = np.zeros(n_classes+1)
        for c in range(n_classes+1):
            TP = conf_matrix[c, 1, 1]
            TN = conf_matrix[c, 0, 0]
            FP = conf_matrix[c, 0, 1]
            FN = conf_matrix[c, 1, 0]
            metric_vector[c] = (TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0
    else:
        raise ValueError("Invalid way argument. Use 'f1_accuracy' or 'accuracy_accuracy'.")
    metric = np.linalg.norm(metric_vector)/np.sqrt(n_classes+1)
    return conf_matrix, metric

_ ,cp_ucem = UCEM(cp.predict_set(alphas=alphas_test,confidence=1-epsilon), y_test)
_ ,cp_mia_ucem = UCEM(cp_mia.predict_set(alphas=alphas_test_mica,confidence=1-epsilon), y_test)

print()
print("CP UCEM:", cp_ucem)
print("CP-MIA UCEM:", cp_mia_ucem)

# Regression

In [5]:
from ucimlrepo import fetch

from base_conformal import ConformalRegressorMIA
from sklearn.model_selection import train_test_split
from crepes import ConformalRegressor
from crepes.extras import DifficultyEstimator
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from pprint import pprint
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing

RANDOM_SEED = 0
epsilon = 0.05

X,y = fetch_california_housing(return_X_y=True)
X = StandardScaler().fit_transform(X)
model = GradientBoostingRegressor(random_state=RANDOM_SEED)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)
X_train, X_calib, y_train, y_calib = train_test_split(X_train, y_train, test_size=0.2, random_state=RANDOM_SEED)

model.fit(X_train, y_train)
y_hat_calib = model.predict(X_calib)
residuals_calib = y_calib - y_hat_calib

y_hat_test = model.predict(X_test)

cr_std = ConformalRegressor()
cr_std.fit(residuals_calib)

de_knn = DifficultyEstimator()
de_knn.fit(X_train, scaler=True)
sigmas_cal_knn_dist = de_knn.apply(X_calib)
cr_norm_knn_dist = ConformalRegressor()
cr_norm_knn_dist.fit(residuals_calib, sigmas=sigmas_cal_knn_dist)
sigmas_test_knn_dist = de_knn.apply(X_test)

de_knn_std = DifficultyEstimator()
de_knn_std.fit(X=X_train, y=y_train, scaler=True)
sigmas_cal_knn_std = de_knn_std.apply(X_calib)
cr_norm_knn_std = ConformalRegressor()
cr_norm_knn_std.fit(residuals_calib, sigmas=sigmas_cal_knn_std)
sigmas_test_knn_std = de_knn_std.apply(X_test)

mia_estimator = ConformalRegressorMIA()
mia_estimator.fit_mia(X_train, y_train)
mia_estimator.fit(residuals_calib,sigmas=mia_estimator.mia.predict(X_calib))
sigmas_test_mia = mia_estimator.mia.predict(X_test)

pprint(f"CP: {cr_std.evaluate(y_hat_test, y_test,y_min=0.0,y_max=1.0,confidence=1-epsilon)}")
pprint(f"CP kNN-Dist: {cr_norm_knn_dist.evaluate(y_hat_test, y_test,sigmas = sigmas_test_knn_dist,y_min=0.0,y_max=1.0,confidence=1-epsilon)}")
pprint(f"CP kNN-Std: {cr_norm_knn_std.evaluate(y_hat_test, y_test,sigmas = sigmas_test_knn_std,y_min=0.0,y_max=1.0,confidence=1-epsilon)}")
pprint(f"CP MIA: {mia_estimator.evaluate(y_hat_test, y_test,sigmas = sigmas_test_mia,y_min=0.0,y_max=1.0,confidence=1-epsilon)}")

def winkler_score(y_true, lower, upper, epsilon=0.05):
    """
    Computes the Winkler Score (Interval Score).
    Lower is better.
    """
    # width penalty
    upper = np.array(upper)
    lower = np.array(lower)
    score = upper - lower
    # under-coverage penalties
    # if y < lower, add 2/epsilon * (lower - y)
    mask_below = y_true < lower
    score[mask_below] += (2.0 / epsilon) * (lower[mask_below] - y_true[mask_below])
    # if y > upper, add 2/epsilon * (y - upper)
    mask_above = y_true > upper
    score[mask_above] += (2.0 / epsilon) * (y_true[mask_above] - upper[mask_above])
    return np.mean(score)

print()
pprint(f"CP Winkler Score: {winkler_score(y_test, cr_std.predict_int(y_hat_test, sigmas=None,confidence=1-epsilon)[:, 0], cr_std.predict_int(y_hat_test, sigmas=None,confidence=1-epsilon)[:, 1])}")
pprint(f"CP kNN-Dist Winkler Score: {winkler_score(y_test, cr_norm_knn_dist.predict_int(y_hat_test, sigmas=sigmas_test_knn_dist,confidence=1-epsilon)[:, 0], cr_norm_knn_dist.predict_int(y_hat_test, sigmas=sigmas_test_knn_dist,confidence=1-epsilon)[:, 1])}")
pprint(f"CP kNN-Std Winkler Score: {winkler_score(y_test, cr_norm_knn_std.predict_int(y_hat_test, sigmas=sigmas_test_knn_std,confidence=1-epsilon)[:, 0], cr_norm_knn_std.predict_int(y_hat_test, sigmas=sigmas_test_knn_std,confidence=1-epsilon)[:, 1])}")
pprint(f"CP MIA Winkler Score: {winkler_score(y_test, mia_estimator.predict_int(y_hat_test, sigmas=sigmas_test_mia,confidence=1-epsilon)[:, 0], mia_estimator.predict_int(y_hat_test, sigmas=sigmas_test_mia,confidence=1-epsilon)[:, 1])}")


("CP: {'error': np.float64(0.8255813953488372), 'eff_mean': "
 "np.float64(0.380427842582406), 'eff_med': np.float64(0.2516682625410517), "
 "'ks_test': np.float64(0.6605192925125578), 'time_fit': "
 "8.296966552734375e-05, 'time_evaluate': 0.03711271286010742}")
("CP kNN-Dist: {'error': np.float64(0.8250968992248062), 'eff_mean': "
 "np.float64(0.45269117733066444), 'eff_med': np.float64(0.3290183410368806), "
 "'ks_test': np.float64(0.20469969448231407), 'time_fit': "
 "0.00010919570922851562, 'time_evaluate': 0.035601139068603516}")
("CP kNN-Std: {'error': np.float64(0.8357558139534884), 'eff_mean': "
 "np.float64(0.37220023406654906), 'eff_med': np.float64(0.31094790210915696), "
 "'ks_test': np.float64(0.10514360699805214), 'time_fit': "
 "8.988380432128906e-05, 'time_evaluate': 0.03391003608703613}")
("CP MIA: {'error': np.float64(0.8255813953488372), 'eff_mean': "
 "np.float64(0.380427842582406), 'eff_med': np.float64(0.2516682625410517), "
 "'ks_test': np.float64(0.667591840317